# AIC25 — Multi-Camera Tracking Pipeline
**Subproject 1: Single-Camera Tracking Consistency**

Everything is cached to Google Drive (2 TB). First session is slow. Every session after is fast.

Run cells **top to bottom**. Skip **[A]** unless you need to retrain the detector.

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    REPO  = '/content/repo'
    PY    = 'python'
    DRIVE = '/content/drive/MyDrive/AIC25'

    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')

    for d in ['models', 'pip_cache', 'outputs/Detection',
              'outputs/EmbedFeature', 'outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)

    os.environ['PIP_CACHE_DIR'] = f'{DRIVE}/pip_cache'
    print(f'Environment : Colab  |  Drive : {DRIVE}')
else:
    REPO  = '/home/seco/deepLearning/Single-Camera-Tracking-Consistency'
    PY    = f'{REPO}/.venv/bin/python'
    DRIVE = None
    os.chdir(REPO)
    print('Environment : Local')

print(f'REPO : {REPO}')

---
## Step 1 — Clone repo + install dependencies *(Colab only)*

In [ ]:
if ON_COLAB:
    pip = f'--cache-dir {DRIVE}/pip_cache'

    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
        print('Repo cloned.')
    else:
        os.system(f'git -C {REPO} pull')
        print('Repo updated.')
    os.chdir(REPO)

    print('Installing BoT-SORT...')
    os.system(f'pip install -q {pip} -r {REPO}/BoT-SORT/requirements.txt')
    os.system(f'pip install -q {pip} cython_bbox pycocotools faiss-gpu cmake')
    os.chdir(f'{REPO}/BoT-SORT')
    os.system('python setup.py develop --quiet')

    print('Installing torchreid...')
    os.system(f'pip install -q {pip} -r {REPO}/deep-person-reid/requirements.txt')
    os.chdir(f'{REPO}/deep-person-reid')
    os.system('python setup.py develop --quiet')

    os.chdir(REPO)
    os.system(f'pip install -q {pip} -r {REPO}/tracking/requirements.txt')
    print('All dependencies installed.')
else:
    print('Local: using .venv — skipping.')

---
## Step 2 — Check GPU

In [ ]:
import subprocess
r = subprocess.run(
    [PY, '-c',
     'import torch; print("CUDA:", torch.cuda.is_available()); '
     'print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")'],
    capture_output=True, text=True
)
print(r.stdout.strip())

if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError(
        '\n\n  ❌  NO GPU DETECTED — pipeline requires a GPU.\n'
        '  Go to:  Runtime → Change runtime type → T4 GPU\n'
        '  Then:   Runtime → Restart session\n'
        '  Then re-run from Step 0.\n'
    )

---
## Step 3 — Download pretrained models *(Colab only)*

In [ ]:
if ON_COLAB:
    os.system('pip install -q gdown')
    M = f'{DRIVE}/models'

    def get_model(local, on_drive, gdrive_id, name):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        if os.path.exists(local):
            print(f'{name}: already local')
        elif os.path.exists(on_drive):
            os.system(f'cp "{on_drive}" "{local}"')
            print(f'{name}: copied from Drive')
        else:
            print(f'{name}: downloading...')
            os.system(f'gdown "https://drive.google.com/uc?id={gdrive_id}" -O "{on_drive}"')
            os.system(f'cp "{on_drive}" "{local}"')
            print(f'{name}: done (cached to Drive)')

    get_model(
        f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar',
        f'{M}/osnet_ms_m_c.pth.tar',
        '1IosIFlLiulGIjwW3H8uMCC3YvMyr9gZ2', 'OSNet'
    )
    get_model(
        f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar',
        f'{M}/bytetrack_x_mot17.pth.tar',
        '1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', 'ByteTrack'
    )

    aic25_drive = f'{M}/ai_city_ckpt.pth.tar'
    aic25_local = f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'
    if os.path.exists(aic25_drive):
        os.system(f'cp "{aic25_drive}" "{aic25_local}"')
        print('AIC25 model: copied from Drive')
    else:
        print('AIC25 model: not trained yet — ByteTrack base model will be used')
else:
    print('Local: models already in place.')

---
## Step 4 — Configure scene
Change `SCENE` here to switch to a different warehouse.

In [ ]:
SCENE   = 'Warehouse_016'
DATASET = 'Val'            # 'Val' (development) | 'Train' (for training)

os.chdir(REPO)
local_data = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
print(f'Scene  : {SCENE}')
print(f'Split  : {DATASET}')
print(f'Exists : {os.path.exists(local_data)}')

---
## Step 5 — Download dataset
*(Colab only — skipped locally)*

> **First session:** downloads ~7 GB from HuggingFace → saved permanently to Drive  
> **Next sessions:** loads from Drive (~2-5 min) — no re-download

**One-time setup:** add your HuggingFace token to Colab Secrets:  
Left panel → 🔑 key icon → **Add secret** → Name: `HF_TOKEN`, Value: your token  
*(Get token at huggingface.co/settings/tokens — needs "read" access)*

Also make sure you have accepted the dataset terms at:  
`huggingface.co/datasets/nvidia/PhysicalAI-SmartSpaces`

In [ ]:
if ON_COLAB:
    import shutil, getpass
    os.system('pip install -q huggingface_hub')
    from huggingface_hub import snapshot_download, login

    drive_data  = f'{DRIVE}/datasets/{DATASET}/{SCENE}'
    local_data  = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    drive_videos = f'{drive_data}/videos'

    os.makedirs(drive_data, exist_ok=True)
    os.makedirs(local_data, exist_ok=True)

    print(f'Drive path   : {drive_data}', flush=True)
    print(f'Local path   : {local_data}', flush=True)
    n_cached = len(os.listdir(drive_videos)) if os.path.exists(drive_videos) else 0
    print(f'Drive videos : {os.path.exists(drive_videos)} ({n_cached} items)', flush=True)

    if os.path.exists(drive_videos) and os.listdir(drive_videos):
        print('[CACHE HIT] Videos already on Drive — skipping download.', flush=True)
    else:
        # Get HF token — Colab Secrets first, then prompt (no widget)
        hf_token = None
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
            print('HF token : loaded from Colab Secrets.', flush=True)
        except Exception:
            pass
        if not hf_token:
            print('HF token not found in Colab Secrets.', flush=True)
            print('Add it: left panel → 🔑 key icon → HF_TOKEN', flush=True)
            hf_token = getpass.getpass('Paste HuggingFace token and press Enter: ')

        login(token=hf_token)
        print('Logged in to HuggingFace.', flush=True)

        hf_split = DATASET.lower()
        print(f'\nDownloading {SCENE} ({DATASET}) — may take 20-40 min...', flush=True)
        snapshot_download(
            repo_id='nvidia/PhysicalAI-SmartSpaces',
            repo_type='dataset',
            local_dir='/content/hf_tmp',
            allow_patterns=[
                f'MTMC_Tracking_2025/{hf_split}/{SCENE}/videos/**',
                f'MTMC_Tracking_2025/{hf_split}/{SCENE}/calibration.json',
                f'MTMC_Tracking_2025/{hf_split}/{SCENE}/ground_truth.json',
            ]
        )
        print('Download finished.', flush=True)

        hf_src = f'/content/hf_tmp/MTMC_Tracking_2025/{hf_split}/{SCENE}'
        print(f'HF source exists : {os.path.exists(hf_src)}', flush=True)
        if os.path.exists(hf_src):
            print(f'HF source contents: {os.listdir(hf_src)}', flush=True)
        else:
            tmp_contents = os.listdir('/content/hf_tmp') if os.path.exists('/content/hf_tmp') else 'missing'
            raise RuntimeError(
                f'Download succeeded but expected path not found: {hf_src}\n'
                f'/content/hf_tmp contents: {tmp_contents}'
            )

        print('Saving to Drive (permanent)...', flush=True)
        if not os.path.exists(drive_videos):
            shutil.copytree(f'{hf_src}/videos', drive_videos)
            print(f'Videos saved — {len(os.listdir(drive_videos))} cameras.', flush=True)
        for fname in ['calibration.json', 'ground_truth.json']:
            src = f'{hf_src}/{fname}'
            if os.path.exists(src):
                shutil.copy(src, f'{drive_data}/{fname}')
                print(f'  {fname} saved.', flush=True)
        shutil.rmtree('/content/hf_tmp', ignore_errors=True)
        print('Temp files cleaned.', flush=True)

    # Link Drive data into repo
    print('\nSetting up local access...', flush=True)
    for fname in ['calibration.json', 'ground_truth.json']:
        src, dst = f'{drive_data}/{fname}', f'{local_data}/{fname}'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  Copied {fname}', flush=True)

    videos_local = f'{local_data}/videos'
    if not os.path.exists(videos_local):
        os.symlink(drive_videos, videos_local)
        print(f'  Symlinked videos/ → Drive', flush=True)
    else:
        print(f'  videos/ already linked', flush=True)
    os.makedirs(f'{local_data}/depth_map', exist_ok=True)

    # Final check
    if os.path.exists(videos_local):
        cameras = sorted(
            c for c in os.listdir(videos_local)
            if os.path.isdir(f'{videos_local}/{c}') and 'map' not in c
        )
        print(f'\n✓ Dataset ready — {len(cameras)} cameras: {cameras}', flush=True)
    else:
        raise RuntimeError(f'[ERROR] videos/ not found at {videos_local}')
else:
    print('Local: using existing AIC25_Track1/ data.')

---
## Link outputs to Drive
Ensures Detection / EmbedFeature / Tracking are written directly to Drive.

In [ ]:
if ON_COLAB:
    import shutil as _shutil
    for folder in ['Detection', 'EmbedFeature', 'Tracking']:
        drive_folder = f'{DRIVE}/outputs/{folder}'
        repo_folder  = f'{REPO}/{folder}'
        os.makedirs(drive_folder, exist_ok=True)
        if os.path.islink(repo_folder):
            pass  # already linked
        elif os.path.isdir(repo_folder):
            _shutil.move(repo_folder, drive_folder)
            os.symlink(drive_folder, repo_folder)
        else:
            os.symlink(drive_folder, repo_folder)
        print(f'{folder}/ → Drive')
    print('Output folders linked to Drive.')
else:
    print('Local: outputs saved directly in repo.')

---
## [A] Train detector
**SKIP THIS CELL** if `ai_city_ckpt.pth.tar` already exists on Drive.  
Only needed once. Takes ~6–10 hours on T4 GPU.

In [ ]:
aic25_drive = f'{DRIVE}/models/ai_city_ckpt.pth.tar' if DRIVE else None

if aic25_drive and os.path.exists(aic25_drive):
    print('✓ AIC25 model already on Drive — training not needed.')
elif not os.path.exists(f'{REPO}/AIC25_Track1/Train'):
    print('⚠ No Train data found — skipping [A].')
    print('  Download the Train split first if you want to train.')
else:
    os.chdir(REPO)
    if DRIVE:
        os.makedirs(f'{DRIVE}/YOLOX_outputs', exist_ok=True)
        os.system(f'ln -sfn {DRIVE}/YOLOX_outputs {REPO}/YOLOX_outputs')

    print('[A] Extracting Train frames...')
    os.system(f'{PY} tools/extract_frames_25.py ./AIC25_Track1/Train -s {SCENE}')

    print('[A] Generating COCO annotations...')
    os.system(f'{PY} tools/convert_to_coco_25.py --BASE_DIR ./AIC25_Track1 -s Train')

    print('[A] Training YOLOX (~6-10 h)...')
    ret = os.system(
        f'{PY} BoT-SORT/yolox/train.py '
        f'-f BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py '
        f'-d 1 -b 4 --fp16 '
        f'-c BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'
    )
    if ret == 0:
        best = f'{REPO}/YOLOX_outputs/yolox_x_AI_City_25/best_ckpt.pth.tar'
        if os.path.exists(best) and DRIVE:
            os.system(f'cp "{best}" "{DRIVE}/models/ai_city_ckpt.pth.tar"')
            os.system(f'cp "{best}" "{REPO}/BoT-SORT/ai_city_ckpt.pth.tar"')
            print('✓ Training done — model saved to Drive.')
    else:
        print(f'[ERROR] Training failed (exit {ret})')

---
## [B+C] Extract frames → Detect → Free disk (per camera)
Processes one camera at a time.  
Frames are cached on Drive so future sessions skip extraction.

In [ ]:
import shutil, cv2
os.chdir(REPO)

local_videos = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'
if not os.path.exists(local_videos):
    print(f'[ERROR] Videos not found: {local_videos}')
    print('       Run Step 5 first.')
else:
    cameras = sorted(
        c for c in os.listdir(local_videos)
        if os.path.isdir(f'{local_videos}/{c}') and 'map' not in c
    )
    if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
        ckpt     = 'BoT-SORT/ai_city_ckpt.pth.tar'
        exp_file = 'BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'
    else:
        ckpt     = 'BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'
        exp_file = 'BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'  # 1-class, matches MOT17 weights
    drive_frames = f'{DRIVE}/frames/{DATASET}/{SCENE}' if DRIVE else None

    print(f'Cameras   : {len(cameras)} — {cameras}')
    print(f'Checkpoint: {ckpt}')
    print(f'Exp file  : {exp_file}\n')

    for i, cam in enumerate(cameras):
        print(f'\n{"="*55}')
        print(f'  Camera {i+1}/{len(cameras)}: {cam}')
        print(f'{"="*55}')

        frame_dir = f'{local_videos}/{cam}/Frame'
        det_txt   = f'{REPO}/Detection/{SCENE}/{cam}.txt'   # detection output is a .txt file, not a folder

        if os.path.exists(det_txt):
            print(f'[SKIP] Detection already done for {cam}')
            continue

        # [B] Frames — from Drive cache or extract from video
        if not (os.path.exists(frame_dir) and os.listdir(frame_dir)):
            drive_cam = f'{drive_frames}/{cam}/Frame' if drive_frames else None
            if drive_cam and os.path.exists(drive_cam) and os.listdir(drive_cam):
                print(f'[B] Copying frames from Drive...')
                os.makedirs(os.path.dirname(frame_dir), exist_ok=True)
                os.system(f'cp -r "{drive_cam}" "{frame_dir}"')
                print(f'[B] Frames ready')
            else:
                print(f'[B] Extracting frames from video...')
                os.makedirs(frame_dir, exist_ok=True)
                mp4 = next(
                    (f'{local_videos}/{cam}/{f}'
                     for f in os.listdir(f'{local_videos}/{cam}') if f.endswith('.mp4')),
                    None
                )
                if not mp4:
                    print(f'[ERROR] No .mp4 in {local_videos}/{cam}')
                    continue
                cap = cv2.VideoCapture(mp4)
                total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                n = 1
                ok, frm = cap.read()
                while ok:
                    cv2.imwrite(f'{frame_dir}/{str(n).zfill(6)}.jpg', frm)
                    ok, frm = cap.read()
                    if n % 1000 == 0:
                        print(f'  {n}/{total} frames', flush=True)
                    n += 1
                cap.release()
                print(f'[B] Extracted {n-1} frames')
                if drive_cam:
                    print(f'  Caching frames to Drive...')
                    os.makedirs(os.path.dirname(drive_cam), exist_ok=True)
                    os.system(f'cp -r "{frame_dir}" "{drive_cam}"')
                    print(f'  Cached.')
        else:
            print(f'[B] Frames already local')

        # [C] Detection — run from REPO root
        print(f'[C] Running detection...')
        os.chdir(REPO)
        ret = os.system(
            f'{PY} BoT-SORT/tools/aic25_get_detection.py '
            f'--scene {SCENE} --dataset {DATASET} --camera {cam} '
            f'-f {exp_file} -c {ckpt} ./'
        )
        if ret != 0:
            print(f'[ERROR] Detection failed (exit {ret}) — check output above for details')
            continue
        print(f'[C] Detection done')

        # Free local disk (frames are on Drive)
        shutil.rmtree(frame_dir, ignore_errors=True)
        print(f'[FREE] Local frames deleted')

    print('\n[DONE] All cameras processed.')

---
## [D] Re-ID embeddings — OSNet
Extracts appearance features from detected object crops.

In [ ]:
os.chdir(f'{REPO}/deep-person-reid')
ret = os.system(f'{PY} torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../')
print('Done' if ret == 0 else f'ERROR {ret}')

---
## [E] Single-camera tracking — all cameras
Runs BoT-SORT on each camera independently.

In [ ]:
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted(d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}') and 'map' not in d)
print(f'Tracking {len(cameras)} cameras...\n')
for cam in cameras:
    print(f'--- {cam} ---')
    ret = os.system(f'{PY} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')

---
## [F] Fix single-camera results
Removes short tracks, fills gaps, applies NMS.

In [ ]:
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted(d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}') and 'map' not in d)
for cam in cameras:
    print(f'--- {cam} ---')
    ret = os.system(f'{PY} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')

---
## [G] Multi-camera tracking
Assigns global IDs across all cameras (Glance + progressive association).

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PY} BoT-SORT/multi_camera_revised.py -s {SCENE} --dataset {DATASET}')
print('Done' if ret == 0 else f'ERROR {ret}')

ret = os.system(f'{PY} BoT-SORT/multi_camera_fix.py -s {SCENE} --dataset {DATASET}')
print('Done' if ret == 0 else f'ERROR {ret}')

---
## [H] Evaluate — HOTA3D
Computes HOTA3D score against ground truth.

In [ ]:
os.chdir(f'{REPO}/TrackEval')
ret = os.system(f'{PY} prepare_eval_data.py -s {SCENE} --exp exp1 --base_dir {REPO}')
print('GT prepared' if ret == 0 else f'ERROR {ret}')

ret = os.system(f'{PY} main.py -s {SCENE} --exp exp1')
print('Evaluation complete' if ret == 0 else f'ERROR {ret}')

---
## Check results

In [ ]:
import json
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted(d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}') and 'map' not in d)

print(f'{"Camera":<12} {"Frames":>8} {"Tracklets":>10}')
print('-' * 32)
for cam in cameras:
    fixed = f'Tracking/Singlecamera/{SCENE}/{cam}/fixed_{cam}.json'
    if os.path.exists(fixed):
        with open(fixed) as f:
            data = json.load(f)
        ids = {t.get('object sc id') for tracks in data.values() for t in tracks}
        print(f'{cam:<12} {len(data):>8} {len(ids):>10}')
    else:
        print(f'{cam:<12}   not yet run')